# Systematics: GENIE

GENIE systematics are handled differently for uncertainty on the event rates vs. uncertainty on the xsec measurement

For the latter, we only consider the impact on response for the signal channel

In [ ]:
%load_ext autoreload
%autoreload 2

#print all output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
from os import path, makedirs
import sys
import uproot
from tqdm import tqdm
import datetime

# local imports
sys.path.append('../../')
from analysis_village.numucc_1p0pi.selection_definitions import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.makedf.util import *
from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from pyanalib.split_df_helpers import *
from pyanalib.stat_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.variable_calculator import get_cc1p0pi_tki
from pyanalib.pandas_helpers import pad_column_name
from makedf.constants import *
from makedf.geniesyst import *

plt.style.use("presentation.mplstyle")
cmap = mpl.cm.viridis
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredText
from matplotlib.offsetbox import AnchoredOffsetbox, DrawingArea, HPacker, VPacker, TextArea
from matplotlib.legend import Legend

# turn off PerformanceWarning
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = False # save covariance matrix
save_fig = save_result # save plots

today_str = datetime.datetime.now().strftime("%Y%m%d")
save_fig_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-genie-{}".format(today_str)
if not path.exists(save_fig_dir):
    makedirs(save_fig_dir)

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"

## -- MC 
# mc_file = path.join(file_dir, "MC", "BNB_cosmics", "genie_wgts-CCQE", "aa_geniewgts_CCQE.df")
mc_file = path.join(file_dir, "MC", "BNB_cosmics", "aa_sel_mup-geniewgts.df")
mc_split_df = pd.read_hdf(mc_file, key="split")
mc_n_split = get_n_split(mc_file)
print("mc_n_split: %d" %(mc_n_split))
print_keys(mc_file)

n_max_concat = 3
mc_keys2load = ['hdr', "mcnu", 'evt'] 
mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
mc_hdr_df = mc_dfs['hdr']
mc_nu_df = mc_dfs['mcnu']
mc_evt_df = mc_dfs['evt']

In [ ]:
## total pot
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))

# target_pot = 1e20
target_pot = mc_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))

mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))
mc_nu_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_nu_df))

In [ ]:
# add multiindex column index "mc" so that branch names match evt_df
# TODO: fix so I don't have to do this
mc_nu_df.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in mc_nu_df.columns])     # match # of column levels

In [ ]:
print("==== selected events ====")
mc_evt_df.loc[:,'nuint_categ'] = get_int_category(mc_evt_df)
mc_evt_df.loc[:,'genie_categ'] = get_genie_category(mc_evt_df)
print(mc_evt_df.nuint_categ.value_counts())
print(mc_evt_df.genie_categ.value_counts())

print("==== all events ====")
mc_nu_df.loc[:,'nuint_categ'] = get_int_category(mc_nu_df)
mc_nu_df.loc[:,'genie_categ'] = get_genie_category(mc_nu_df)
print(mc_nu_df.nuint_categ.value_counts())
print(mc_nu_df.genie_categ.value_counts())


In [ ]:
syst_name = "GENIE"
mc_evt_df[("mc", syst_name)]

In [ ]:
var_config = VariableConfig.muon_momentum()
ret_dists = get_distributions(mc_evt_df, mc_nu_df, var_config)

In [ ]:
nevts_signal_truth, _, _     = plt.hist(ret_dists["var_truth_signal"],     bins=var_config.bins, weights=ret_dists["weight_truth_signal"], histtype="step", label="All Signal in MC")
nevts_signal_sel_truth, _, _ = plt.hist(ret_dists["var_signal_sel_truth"], bins=var_config.bins, weights=ret_dists["weight_signal"],       histtype="step", label="Selected Signal, True")
nevts_signal_sel_reco, _, _  = plt.hist(ret_dists["var_signal_sel_reco"],  bins=var_config.bins, weights=ret_dists["weight_signal"],       histtype="step", label="Selected Signal, Reco", color="k")

plt.legend()
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Events / Bin")
plt.xlim(var_config.bins[0], var_config.bins[-1])

if save_fig:
    plt.savefig("{}/{}-sel_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show();

In [ ]:
# Note: these are background subtracted
univ_events, cv_events = get_genie_univs("xsec", mc_evt_df, mc_nu_df, var_config, syst_name)
ret_xsec = get_covariance(univ_events, cv_events)
plot_univ_hist(mc_evt_df, var_config, ("mc", syst_name), univ_events, cv_events)

univ_events, cv_events = get_genie_univs("rate", mc_evt_df, mc_nu_df, var_config, syst_name)
ret_rate = get_covariance(univ_events, cv_events)
plot_univ_hist(mc_evt_df, var_config, ("mc", syst_name), univ_events, cv_events)

In [ ]:
matrix_type = "Covariance"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret_xsec[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)
plot_heatmap(ret_rate[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
frac_unc = np.sqrt(np.diag(ret_xsec["Covariance_Frac"]))
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_unc, histtype="step", color="black")
frac_unc = np.sqrt(np.diag(ret_rate["Covariance_Frac"]))
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_unc, histtype="step", color="gray")

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Fractional Uncertainty")
plt.grid(True)
plt.show();

In [ ]:
if 'syst_dict' in locals():
    syst_dict[var_config.var_save_name] = ret_xsec["Covariance_Frac"]
    syst_dict[var_config.var_save_name+"_rate"] = ret_rate["Covariance_Frac"]

else:
    syst_dict = {}
    syst_dict[var_config.var_save_name] = ret_xsec["Covariance_Frac"]
    syst_dict[var_config.var_save_name+"_rate"] = ret_rate["Covariance_Frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/genie_syst_dict.npz", **syst_dict)

# sanity check: load saved file and check it agrees with the local one for all keys
loaded = np.load(file_dir + "/genie_syst_dict.npz")
for key in syst_dict.keys():
    arr_local = syst_dict[key]
    arr_loaded = loaded[key]
    assert np.allclose(arr_local, arr_loaded), f"Mismatch for key: {key}"
print("All keys in syst_dict match the saved npz file.")